# Naija-Switch -- LoRA fine-tune (Colab)

Runs the real fine-tuning job (not the CPU smoke test) from the
[Naija-Code-Switch](https://github.com/ayoolaeni/Naija-Code-Switch) repo
against a free Colab GPU.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then
`Runtime -> Run all`. No Hugging Face token or license click-through is
needed -- the base model below is fully open, ungated.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU detected -- go to Runtime > Change runtime type > T4 GPU, then Run all again. Training will still run on CPU but much slower.'

## Why this base model

Short version: **no openly available conversational LLM is actually
pretrained on real Nigerian Pidgin.** The only public models that have
genuinely seen Pidgin text -- AfriBERTa, AfriTeVa, the MasakhaNER-tuned
RoBERTa (`arnolfokam/roberta-base-pcm`) -- are small encoder/NER/classification
models, not instruction-following chat models, so they're a poor starting
point for a chatbot. NLLB was once planned to cover Pidgin (`pcm_Latn`) but
it never shipped in the released FLORES-200 language set either.

So the practical choice -- and the one already configured in this repo's
`configs/training_config.yaml` -- is a strong general-purpose
instruction-tuned model with a good multilingual tokenizer:
**Qwen/Qwen2.5-1.5B-Instruct**.

- **Ungated.** Downloads with no token or license click-through -- matters
  for a notebook meant to just be run end-to-end. Llama 3 and Gemma 2 both
  require accepting a license on Hugging Face first.
- **Small.** 1.5B parameters fits and fine-tunes comfortably on a free
  Colab T4 in a few minutes with plain LoRA -- no 4-bit/QLoRA needed at
  this size (the repo's `train_lora.py` still supports QLoRA if you swap
  to a bigger base model later).
- **Low token fragmentation on Pidgin, incidentally.** Pidgin is
  English-lexified, so most Pidgin words tokenize the same as their English
  spellings in Qwen's vocabulary -- this is the real reason fine-tuning
  isn't painful here, more than any claimed native Pidgin knowledge.
- **Already knows how to converse.** Being instruction-tuned, it can hold
  a coherent multi-turn exchange out of the box; LoRA's job here is to
  shift its *style* toward English-Pidgin mixing, not teach conversation
  from zero.

If you want to compare against Llama 3.2 or Gemma 2 later, run
`python -m src.modeling.baseline_probe` (needs `HF_TOKEN` + accepting each
model's license) -- that script already builds exactly this comparison.

In [ ]:
import os

REPO_URL = "https://github.com/ayoolaeni/Naija-Code-Switch.git"
REPO_DIR = "Naija-Code-Switch"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q -r requirements.txt

# Colab's base image sometimes ships an old torchao. Recent peft versions
# probe for torchao and hard-error if it's present but below 0.16.0 --
# even though this project never imports torchao at all (LoRA doesn't
# need it). Simplest fix: remove it so the probe is skipped.
!pip uninstall -y -q torchao

In [ ]:
# data/processed/ and data/splits/ are gitignored on purpose (derived
# data doesn't belong in git) -- a fresh clone only has the raw sources
# under data/authored/ and data/raw/. Regenerate the processed + split
# files from those before anything below can read data/splits/*.jsonl.
!python -m src.data_pipeline.pipeline
!python -m src.data_pipeline.split

In [ ]:
for split in ["train", "val", "test"]:
    path = f"data/splits/{split}.jsonl"
    n = sum(1 for _ in open(path, encoding="utf-8"))
    print(f"{split}: {n} dialogues")

## 1. Smoke test first

Proves the training code path (LoRA injection, tokenization, loss masking,
the overfitting check) is correct, on CPU, in seconds, using a tiny
placeholder model -- before spending real GPU time on the actual base
model. Safe to skip on repeat runs once you've seen it pass.

In [ ]:
!python -m src.modeling.train_lora --smoke-test

## 2. Real fine-tune

Uses `configs/training_config.yaml` as-is: Qwen2.5-1.5B-Instruct, LoRA
rank 16 on the attention Q/V projections, learning rate 2e-4, up to 4
epochs with early stopping on validation loss.

**Honest caveat:** most of the training set is script-generated (from
`scripts/generate_synthetic_dialogues.py`), not organically collected from
many real human code-switchers as the research brief intends -- see the
data-check cell above for the current split sizes, and `README.md`'s
"Scope of this build" section for the full breakdown. This is enough to
prove the real GPU training path works end-to-end and produces a loadable
adapter that has actually learned the style in the data -- it is not the
same as training on a real-contributor-authored corpus of this size.
Watch the console for the `[overfitting-guardrail] WARNING` line at the
end; with a repetitive templated batch mixed in, some near-verbatim
warnings are still possible and not necessarily a bug.

In [ ]:
!python -m src.modeling.train_lora --config configs/training_config.yaml

## 3. Adapter auto-downloads here

Colab wipes its disk when the runtime recycles, so the cell right below
zips the trained adapter and triggers a browser download **automatically
as part of Run all** -- you don't need to click or choose anything, the
zip lands in your normal Downloads folder once training finishes.

(An optional Google Drive backup cell follows after it, in case you'd
rather keep a copy there too -- that one does need a one-time Google
sign-in click, so it's kept separate and optional rather than blocking
the automatic download above.)

In [ ]:
# Automatic: zip the trained adapter and download it straight to your
# machine. This cell needs no input -- it just runs as part of Run all.
import shutil
from google.colab import files

shutil.make_archive("naija-switch-lora-adapter", "zip", "checkpoints/naija-switch-lora")
files.download("naija-switch-lora-adapter.zip")

### Optional: also back up to Google Drive

Not required -- the cell above already downloaded the zip to your
computer. Run this only if you want a second copy in Drive too (needs a
one-time click to authorize Google Drive access).

In [ ]:
# Optional: also copy to Google Drive (needs a one-time sign-in click,
# so this is NOT auto-run silently the way the download cell above is --
# Colab will pause here waiting for you to authorize if you run it).
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = "/content/drive/MyDrive/naija-switch-checkpoints/naija-switch-lora"
shutil.copytree("checkpoints/naija-switch-lora", dest, dirs_exist_ok=True)
print("Saved to", dest)

## 4. Quick test -- talk to the fine-tuned model

Loads the base model plus the LoRA adapter you just trained and runs a
few code-switched prompts through it, using the same system prompt the
chat app uses.

In [ ]:
from src.modeling.inference import LocalGenerator, GenerationParams
from src.modeling.prompt_design import get_frozen_system_prompt

generator = LocalGenerator(
    base_model_id="Qwen/Qwen2.5-1.5B-Instruct",
    lora_adapter_dir="checkpoints/naija-switch-lora",
)
system_prompt = get_frozen_system_prompt()

test_inputs = [
    "Abeg how far, wetin dey happen?",
    "I don tire o, work too much today.",
    "Can you help me check my account balance?",
]

for text in test_inputs:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text},
    ]
    reply = generator.generate(messages, GenerationParams(max_new_tokens=80))
    print(f"User: {text}\nNaija-Switch: {reply}\n")

## Next steps

- This run proves the real GPU fine-tuning path end-to-end -- the thing
  that hadn't happened yet before this notebook. The corpus now clears the
  brief's numeric 3,000-5,000 turn target, but ~97% of those turns are
  script-generated rather than from real human code-switchers (see
  `README.md`), so this is still not the brief's intended dataset.
- To genuinely improve the model, replace the synthetic batch with real
  contributor-authored dialogues over time: add to `data/authored/` as
  their own `*.jsonl` file (the loader in
  `src/data_pipeline/sources/authored.py` picks up any file there
  automatically), re-run `python -m src.data_pipeline.pipeline` and
  `python -m src.data_pipeline.split`, then re-run this notebook.
- To compare base models empirically instead of by argument, run
  `python -m src.modeling.baseline_probe` (needs `HF_TOKEN` and accepting
  each gated model's license on Hugging Face for Llama 3 / Gemma 2).
- To evaluate the fine-tuned model against the unadapted baseline, set
  `LOCAL_LORA_ADAPTER_DIR=checkpoints/naija-switch-lora` and run
  `python -m src.evaluation.compare --systems mock local` (space-separated)
  against `data/splits/test.jsonl`.